# 02 — Modélisation EVT du MASI : GEV, POT-GPD, GARCH-EVT
**Achraf Akiyaf — Master MMSD, FST Tanger**

Objectif : modéliser les queues de la distribution des pertes du MASI par la Théorie des Valeurs Extrêmes, et en déduire VaR et Expected Shortfall.

## 0. Chargement (réutilise les modules src/)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data_utils import load_masi, audit_and_clean, log_returns, stylized_facts
from src import evt, margin, backtest as bt
from src.garch_evt import fit_garch_evt
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3})

masi = load_masi('../data/raw/masi.csv')
masi, journal = audit_and_clean(masi)
r = log_returns(masi); losses = -r
print('Période:', masi.index.min().date(),'->',masi.index.max().date(),'|',len(masi),'jours')
print('Corrections:', journal)

## 1. Approche Bloc-Maxima — loi GEV
On prend le pire jour de chaque mois. Théorème de Fisher-Tippett : ces maxima suivent une loi GEV. Le paramètre **ξ** donne la nature de la queue (ξ>0 = Fréchet, queue lourde).

In [ ]:
bm = evt.block_maxima(losses, freq='ME')
gev = evt.fit_gev(bm)
print(f'Blocs (mois) : {gev.n_blocks}')
print(f'xi = {gev.xi:.3f}  |  mu = {gev.mu:.3f}  |  sigma = {gev.sigma:.3f}')
print(f'Famille : {gev.family}')
print(f'Test de Gumbel (H0: xi=0) : p-value = {gev.lr_pvalue_gumbel:.4f}')

plt.figure(figsize=(11,3.5))
plt.plot(losses.index, losses.values, lw=0.4, color='gray', label='Pertes')
plt.scatter(losses.resample('ME').max().dropna().index,
            losses.resample('ME').max().dropna().values,
            color='red', s=18, zorder=5, label='Maxima mensuels')
plt.title('Bloc-Maxima'); plt.ylabel('Perte (%)'); plt.legend(); plt.show()

## 2. Choix du seuil (l'étape clé du POT)
Deux diagnostics : le **Mean Residual Life plot** (doit devenir linéaire) et le **Parameter Stability plot** (ξ doit se stabiliser). On choisit u au début de la zone stable.

In [ ]:
mrl = evt.mean_residual_life(losses.values)
ps = evt.parameter_stability(losses.values)
u = evt.suggest_threshold(losses.values)
print(f'Seuil suggéré : u = {u:.2f}%')

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(mrl.u, mrl.mrl, color='#2e75b6')
ax[0].fill_between(mrl.u, mrl.lo, mrl.hi, alpha=0.2)
ax[0].axvline(u, color='red', ls='--'); ax[0].set_title('Mean Residual Life'); ax[0].set_xlabel('u')
ax[1].plot(ps.u, ps.xi, 'o-', color='#c55a11', ms=3)
ax[1].axvline(u, color='red', ls='--'); ax[1].set_title('Parameter Stability — xi(u)'); ax[1].set_xlabel('u')
plt.tight_layout(); plt.show()

## 3. Ajustement GPD & mesures de risque
VaR et Expected Shortfall (CVaR) par les formules fermées de McNeil & Frey.

In [ ]:
gpd = evt.fit_gpd(losses, u)
print(f'GPD : xi = {gpd.xi:.3f} | beta = {gpd.beta:.3f} | {gpd.n_exceed} excès')
print(f'Cohérence GEV/GPD : xi_GEV={gev.xi:.2f} vs xi_GPD={gpd.xi:.2f}\n')
print(f"{'Niveau':>8} {'VaR (%)':>10} {'ES/CVaR (%)':>12}")
for a in (0.95, 0.99, 0.999):
    v,e = evt.gpd_var_es(gpd, a)
    print(f'{a:>8.3f} {v:>10.2f} {e:>12.2f}')

# QQ-plot d'adéquation
from scipy.stats import genpareto
exc = np.sort(gpd.exceedances); N=len(exc)
th = genpareto.ppf((np.arange(1,N+1)-.5)/N, c=gpd.xi, scale=gpd.beta)
plt.figure(figsize=(5,5)); plt.scatter(th, exc, s=12, color='#2e75b6')
m=max(th.max(),exc.max()); plt.plot([0,m],[0,m],'r--')
plt.xlabel('Quantiles GPD'); plt.ylabel('Quantiles empiriques'); plt.title('QQ-Plot GPD'); plt.show()

## 4. Return Level Plot avec incertitude bootstrap

In [ ]:
T = np.logspace(np.log10(30), np.log10(3650), 40)
rl = evt.gpd_return_level(gpd, T)
lo, hi = evt.gpd_return_level_ci(gpd, T, n_boot=500)
plt.figure(figsize=(9,4))
plt.fill_between(T, lo, hi, alpha=0.2, color='red', label='IC 95% bootstrap')
plt.plot(T, rl, color='red', label='Niveau de retour')
plt.xscale('log'); plt.xlabel('Période de retour (jours)'); plt.ylabel('Perte (%)')
plt.title('Return Level Plot'); plt.legend(); plt.show()

## 5. GARCH-EVT : VaR dynamique
GJR-GARCH(1,1) pour la volatilité conditionnelle, POT sur les résidus.

In [ ]:
g = fit_garch_evt(r)
print(f'sigma(t+1) prévu = {g.forecast_vol:.2f}% | xi résidus = {g.gpd.xi:.3f}')
plt.figure(figsize=(11,4))
plt.plot(r.index, r.values, lw=0.4, color='gray', label='Rendements')
plt.plot(r.index, -g.var_dyn['VaR_0.990'], color='red', lw=0.9, label='-VaR99 dynamique')
plt.title('VaR 99% conditionnelle (GARCH-EVT)'); plt.legend(); plt.show()

## Conclusion
ξ ≈ 0.3 (Fréchet) confirmé par GEV **et** GPD ; VaR99 ≈ 2%, ES99 ≈ 3.2%. La VaR EVT dépasse nettement la VaR gaussienne du notebook 01 → l'EVT capture correctement le risque extrême. Suite : notebook 03, marge du Future MASI 20.